In [2]:
import pandas as pd
import numpy as np

In [4]:
data = {
    'EmployeeID': list(range(1, 21)),
    'Age': [25, 30, np.nan, 35, 28, 150, 40, 33, 29, 45, 38, 26, 50, 31, 27, np.nan, 42, 36, 34, 39],
    'Gender': [
        'Male', 'Female', 'Male', 'Female', 'Male',
        'Female', np.nan, 'Male', 'Female', 'Male',
        'Female', 'Male', 'Female', 'Male', 'Female',
        'Male', 'Female', 'Male', 'Female', 'Male'],
    'Department': [
        'Sales', 'IT', 'HR', 'Finance', 'IT',
        'Sales', 'HR', 'Finance', 'IT', 'Sales',
        np.nan, 'IT', 'Finance', 'HR', 'Sales',
        'IT', 'Finance', 'Sales', 'HR', 'IT'],
    'Education': [
        'Bachelor', 'Master', 'Bachelor', 'PhD', 'Bachelor',
        'High School', 'Master', 'Bachelor', 'Master',
        'High School', 'PhD', 'Bachelor', 'Master',
        'Bachelor', 'High School', 'Master', 'PhD',
        'Bachelor', 'Master', 'Bachelor'],
    'Experience': [
        2, 5, 3, 10, 4, 1, 12, 8, 5, np.nan,
        15, 3, 20, np.nan, 2, 7, 18, 9, 11, np.nan],
    'Salary': [
        45000, 60000, 50000, 90000, np.nan,
        35000, 75000, 65000, 58000, 48000,
        95000, 9999999, 110000, np.nan, 38000,
        70000, 105000, 62000, 72000, 68000],

    'City': [
        'Mumbai', 'Delhi', 'Mumbai', 'Bangalore', 'Delhi',
        'Pune', 'Mumbai', 'Bangalore', 'Delhi', 'Pune',
        'Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Mumbai',
        'Delhi', 'Bangalore', 'Pune', 'Mumbai', 'Delhi']
}

df = pd.DataFrame(data)
    

In [5]:
print(df.isnull().sum())

EmployeeID    0
Age           2
Gender        1
Department    1
Education     0
Experience    3
Salary        2
City          0
dtype: int64


In [6]:
print((df.isnull().mean()*100).round(1))

EmployeeID     0.0
Age           10.0
Gender         5.0
Department     5.0
Education      0.0
Experience    15.0
Salary        10.0
City           0.0
dtype: float64


In [8]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Department'] = df['Department'].fillna(df['Department'].mode()[0])
df['Experience'] = df['Experience'].fillna(df['Experience'].median())
df['Salary'] = df['Salary'].fillna(df['Salary'].median())

print(df.isnull().sum().sum())


EmployeeID    0
Age           0
Gender        0
Department    0
Education     0
Experience    0
Salary        0
City          0
dtype: int64


In [9]:
print(df.isnull().sum().sum())

0


In [15]:
def iqr_outliers(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    return lower, upper

    for col in ['Age', 'Salary']:
        lo, hi = iqr_outliers(df[col])
        print(col, '-> keep between', lo, 'and', hi)
        
        # Keep only rows inside the bounds for Both Columns
        mask = pd.series(True, index = df.index)
        for col in ['Age', 'Salary']:
            lo, hi = iqr_outliers(df[col])
            mask &= df[col].between(lo, hi)

            df = df[mask].rest_index(drop=True)

In [18]:
# Education: ORDINAL -> label encode with a defined order
edu_order = {
    'High School': 0,
    'Bachelor': 1,
    'Master': 2,
    'PhD': 3
}

df['Education_Label'] = df['Education'].map(edu_order)

# Gender: only two values -> 0/1
df['Gender_Label'] = df['Gender'].map({
    'Male': 0,
    'Female': 1
})

# Department & City: NOMINAL -> one-hot encode
df = pd.get_dummies(
    df,
    columns=['Department', 'City'],
    prefix=['Dept', 'City'])

# (optional) turn True/False dummies into 1/0 integers
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

KeyError: "None of [Index(['Department', 'City'], dtype='object')] are in the [columns]"